# 실습 E: 위키백과 일별 조회수와 반감 일수

『대중천문학개론』 10장 6.1절 실습 E의 코드 전체와 네 문서로 넓힌 확장. `OFFLINE = True`이면 `data/fig10-4.csv`와 `data/fig10-4_baseline.csv`로, `False`이면 Wikimedia 조회수 API에서 새로 받는다.

User-Agent에는 ASCII 글자만 넣고, example.org 주소와 메일을 자기 수업 누리집과 연락처로 바꾼다.

In [ ]:
OFFLINE = True   # 인터넷이 되면 False로 바꾼다

In [ ]:
# Colab에서 열었으면 저장소를 받아 ch10 폴더로 옮긴다(로컬에서는 아무 일도 하지 않는다)
import os, subprocess, sys
if "google.colab" in sys.modules and not os.path.exists("README.md"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/Park-Jiwon-Maker/public-astronomy-textbook.git", "repo"], check=True)
    os.chdir("repo/ch10")
print(os.getcwd())

In [ ]:
if OFFLINE:                      # 인터넷 대신 data/ 폴더의 고정 자료로 응답하게 한다
    import offline_fixtures
    offline_fixtures.install()

## 본문 코드

In [ ]:
# 실습 E: 위키백과 일별 조회수와 반감 일수
import requests
import pandas as pd
from urllib.parse import quote

BASE = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"
# "도구 이름/버전 (연락처)" 형식. 머리글에는 ASCII만 쓰므로 주소와 메일을 영문 그대로 자기 것으로 바꾼다
HEAD = {"User-Agent": "astro-class-lab/0.1 (https://example.org/class; contact@example.org)"}

def daily(project, title, start, end):
    url = f"{BASE}/{project}/all-access/user/{quote(title, safe='')}/daily/{start}/{end}"
    r = requests.get(url, headers=HEAD, timeout=30)
    r.raise_for_status()
    s = pd.Series({it["timestamp"][:8]: it["views"] for it in r.json()["items"]})
    s.index = pd.to_datetime(s.index, format="%Y%m%d")   # UTC 날짜
    return s

def half_days(v, peak):        # 정점의 절반 아래로 처음 떨어지기까지의 날수. 기간 안에 없으면 None
    after = v[v.index > peak]
    below = after[after < v[peak] / 2]
    return (below.index[0] - peak).days if len(below) else None

T = "Solar_eclipse_of_April_8,_2024"
v = daily("en.wikipedia", T, "20240320", "20240430")
peak = v.idxmax()
base = daily("en.wikipedia", T, "20240501", "20240531").median()   # 조용한 달의 중앙값
# 사건 전에 관심이 오르지 않는 문서는 정점 전 30일 중앙값을 쓴다:
# base = v[v.index < peak].tail(30).median()
ratio = round(v[peak] / base, 1) if base > 0 else None   # 기준선이 0이거나 결측이면 None
print(peak.date(), ratio, half_days(v, peak))   # 2024-04-08 1612.6 1

## 네 문서로 넓히기 (실습 E 3–5단계)

In [ ]:
BASE = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"
HEAD = {"User-Agent": "astro-class-lab/0.1 (https://example.org/class; contact@example.org)"}

# 문서 네 개: (프로젝트, 제목, 기준선 기간). 기준선은 사건과 떨어진 조용한 달의 중앙값이다
DOCS = [
    ("en.wikipedia", "Solar_eclipse_of_April_8,_2024", ("20240501", "20240531")),
    ("en.wikipedia", "Solar_eclipse",                  ("20240201", "20240229")),
    ("ko.wikipedia", "2024년_4월_8일_일식",            ("20240501", "20240531")),
    ("ko.wikipedia", "일식",                           ("20240201", "20240229")),
]


def daily(project, title, start, end):
    url = f"{BASE}/{project}/all-access/user/{quote(title, safe='')}/daily/{start}/{end}"
    r = requests.get(url, headers=HEAD, timeout=30)
    r.raise_for_status()
    s = pd.Series({it["timestamp"][:8]: it["views"] for it in r.json()["items"]})
    s.index = pd.to_datetime(s.index, format="%Y%m%d")   # UTC 날짜
    return s


def half_days(v, peak):
    """정점 뒤 처음으로 정점의 절반 아래로 떨어진 날까지의 일수. 관찰 기간 내 미도달이면 None."""
    after = v[v.index > peak]
    below = after[after < v[peak] / 2]
    return (below.index[0] - peak).days if len(below) else None


def ratio(peak_views, base):
    """정점/기준선 배수. 기준선이 0이거나 결측(None, NaN)이면 None."""
    return round(peak_views / base, 1) if pd.notna(base) and base > 0 else None


print("문서 | 정점 날짜(UTC) | 정점/기준선 배수 | 반감 일수 | 절반 넘은 날 수")
series = {}
for project, title, (b0, b1) in DOCS:
    v = daily(project, title, "20240320", "20240430")
    base = daily(project, title, b0, b1).median()
    peak = v.idxmax()
    above = int((v > v[peak] / 2).sum())   # Baram-Tsabari와 Segev(2018)식: 최댓값의 절반을 넘은 날 수
    print(f"{project} {title}", peak.date(), ratio(v[peak], base), half_days(v, peak), above, sep=" | ")
    series[title] = v

# 예외 처리 확인: 정점 날짜에서 끝나는 짧은 창(관찰 기간 내 미도달), 기준선 0과 결측
v = series["Solar_eclipse_of_April_8,_2024"].loc[:"2024-04-08"]
print("짧은 창(3/20–4/8) 반감 일수:", half_days(v, v.idxmax()),
      "| 기준선 0일 때 배수:", ratio(v.max(), 0), "| 기준선 결측일 때 배수:", ratio(v.max(), float("nan")))